# `code/pipeline/p001_48_region_split.py`

Read-only rendering of the script (no outputs; it needs the licensed inputs described in `../../DATA_ACCESS.md`). The .py file is the version of record.


```text
p001_48 — 지역 분할 로버스트니스: 미국 vs 유럽(24개국) — 헤드라인 격차 사다리 · 판별 마진 · 고정지평 구성 계수

[왜] 본분석 표본 NA+EU 는 미국 74%·유럽 22%·캐나다 4% 다. 비교 문헌(GMWX 2022) 이 미국 표본이라 심판은 "미국만으로 서는가, 유럽에서도 같은가" 를 묻는다.
 §3 매칭에는 NA−EU 차이(Table 2)가 있지만 §4 격차·판별과 §6 구성 계수에는 지역 분할이 없다. 캐나다는 4,721 딜로 NA 의 5% 라 미국과 따로 떼지 않는다(US-only ≈ NA).

[구성] sample_v1. 미국 = USA · 유럽 = EU24 (p001_rescue_common.EU). 셀 벤치마크는 지역 안에서 다시 만든다(연도×섹터×단계는 국가 무관이나 표본이 지역별).
 A 헤드라인 사다리 (P001-10 사양: FF 딜, exit_ever, dt ≤ 2017-10; cell_cat / cell_stage 셀 내 fp 계수; 투자사 군집 부트 500) — 지역별 + 혼합 셀 수·딜 수 + 셀 내 재배정 위약 p95 (P001-40 사양)
 B 판별 마진 (FF 딜, cell_stage): fon(36m 후속) · exit3(36m 출구, 사후 ≤2020-10 무관 — 사전기 딜) 의 fp 계수 — 지역별, MDE 병기
 C 고정 출구지평 구성 계수 (P001-33 A F2t 사양: exit3/exit3, 연공 TEN) — 지역별 파트너 패널
[사전 예측] (2026-09-09, 결과 조회 전)
 A: 미국 cell_cat 격차 ∈ [−7, −2]pp (NAEU −4.55 와 유사), 혼합 셀 120–145; 유럽 혼합 셀 25–50, 격차 CI 반폭 ≥ 8pp(부호 불확정). 위약 p95: 미국 5–7, 유럽 ≥ 10.
 B: 미국 fon·exit3 fp 계수 CI 0 포함; 유럽도 0 포함(MDE 더 큼).
 C: 미국 F2t ∈ [0.20, 0.35] 하한 > 0; 유럽 ∈ [−0.1, 0.5] CI 0 포함 (파트너 ~350).
[판정] 로버스트니스 — status OK. 유럽 계수가 미국과 **부호가 다르고 CI 가 겹치지 않으면** verdict 에 "지역 이질성" 표기.
```


In [ ]:
import numpy as np
import pandas as pd

from p001_rescue_common import (COMMON_SHA, CUT, END_FON, EU, TC, TEN, add_post, add_tenure, boot, emit, first_deal_dates, fmt, load_deals, log,
                                partner_pre, qci)

rng = np.random.default_rng(20260948)
NB, NPL = 500, 400
OUT = {}
dn = load_deals(with_exit_dt=True)
first = first_deal_dates()
REG = {"USA": dn["country_code"] == "USA", "NA": dn["country_code"].isin({"USA", "CAN"}), "EU24": dn["country_code"].isin(EU)}
# NA(미국+캐나다) 는 PI 요청으로 추가(2026-09-09, 결과 조회 전): 캐나다 4,721 딜이라 미국과 ±0.5pp 안에서 일치할 것으로 예측.


def within_beta(dd, cell, y="exit_ever"):
    yr = (dd[y] - dd[y].groupby(dd[cell]).transform("mean")).to_numpy()
    xr = (dd["fp"] - dd["fp"].groupby(dd[cell]).transform("mean")).to_numpy()
    sxx = float((xr * xr).sum())
    return float((xr * yr).sum() / sxx) if sxx > 0 else np.nan


def boot_gap(dd, cell, y="exit_ever", nb=NB):
    b = within_beta(dd, cell, y)
    grp = {c: g.index.to_numpy() for c, g in dd.groupby("investor_uuid")}; kl = list(grp); bs = []
    for _ in range(nb):
        pick = rng.integers(0, len(kl), len(kl)); s = dd.loc[np.concatenate([grp[kl[i]] for i in pick])]
        v = within_beta(s, cell, y)
        if np.isfinite(v): bs.append(v)
    lo, hi = qci(np.array(bs)); se = float(np.std(bs, ddof=1))
    return {"coef_pp": round(b * 100, 3), "ci95_pp": [round(lo * 100, 2), round(hi * 100, 2)], "mde80_pp": round(2.8 * se * 100, 2), "n": int(len(dd)), "sig": bool(lo > 0 or hi < 0)}


def placebo_p95(m, cell, y="exit_ever", n=NPL):
    pl = []
    mm = m.reset_index(drop=True)
    yr = (mm[y] - mm[y].groupby(mm[cell]).transform("mean")).to_numpy()
    for _ in range(n):
        fpp = mm.groupby(cell)["fp"].transform(lambda s: pd.Series(rng.permutation(s.to_numpy()), index=s.index))
        xr = (fpp - fpp.groupby(mm[cell]).transform("mean")).to_numpy()
        pl.append(abs(float((xr * yr).sum() / (xr * xr).sum())) * 100)
    return round(float(np.percentile(pl, 95)), 2)


for reg, mask in REG.items():
    log("\n" + "=" * 100 + f"\n[{reg}]\n" + "=" * 100)
    d = dn[mask].copy()
    ffd = d[(d["ff"] == 1) & (d["dt"] <= CUT)].copy()
    ffd["cell_year"] = ffd["year"]
    res = {"n_deals": int(len(d)), "n_ff_pre": int(len(ffd)), "n_female_partners_ff": int(ffd.loc[ffd["fp"] == 1, "partner_uuid"].nunique())}
    # A 사다리
    A = {}
    for name, cell in (("year", "cell_year"), ("invyear", "cell0"), ("pluscat", "cell_cat"), ("plusstage", "cell_stage")):
        g = ffd.groupby(cell)["fp"].agg(["mean", "size"]); mixed = g.index[(g["mean"] > 0) & (g["mean"] < 1)]
        m = ffd[ffd[cell].isin(mixed)]
        r = boot_gap(ffd, cell)
        r.update({"n_mixed_cells": int(len(mixed)), "n_deals_mixed": int(len(m)), "share_fp_deals_with_peer": round(float(ffd.loc[ffd["fp"] == 1, cell].isin(mixed).mean()), 4)})
        if name in ("pluscat", "plusstage") and len(m) >= 50:
            r["placebo_p95_abs_pp"] = placebo_p95(m, cell)
        A[name] = r
        log(f"  A {name:<10} β {r['coef_pp']:+.2f}pp [{r['ci95_pp'][0]:+.2f},{r['ci95_pp'][1]:+.2f}] MDE {r['mde80_pp']:.1f} · 혼합 {r['n_mixed_cells']} 셀 / {r['n_deals_mixed']} 딜 · FP 동료가용 {r['share_fp_deals_with_peer']:.3f}" + (f" · 위약 p95 {r['placebo_p95_abs_pp']}" if "placebo_p95_abs_pp" in r else ""))
    res["A_ladder"] = A
    # B 판별 마진 (cell_stage)
    Bm = {}
    for y in ("fon", "exit3", "exit_ever"):
        r = boot_gap(ffd, "cell_stage", y); Bm[y] = r
        log(f"  B {y:<9} cell_stage fp {r['coef_pp']:+.2f}pp [{r['ci95_pp'][0]:+.2f},{r['ci95_pp'][1]:+.2f}] MDE {r['mde80_pp']:.1f}")
    res["B_adjudication_stage_cell"] = Bm
    # C 고정지평 구성 계수
    P, _ = partner_pre(d, "exit3"); P = add_post(P, d, "exit3", end=END_FON); P = add_tenure(P, first=first)
    C = {"n_partners": int(len(P)), "n_post": int(P["has_post"].sum())}
    C["F2t"] = boot(P, "post_adj", ["terrain", "adj", "ln_n", "fp"] + TEN, ["terrain", "adj"], rng, nb=400)
    C["F2"] = boot(P, "post_adj", TC + ["adj", "ln_n", "fp"] + TEN, TC, rng, nb=400)
    log(f"  C 파트너 {C['n_partners']:,} (사후 {C['n_post']:,}) · F2t terrain {fmt(C['F2t'], 'terrain') if C['F2t'] else 'NA'} · adj {fmt(C['F2t'], 'adj') if C['F2t'] else 'NA'} · t_cs {fmt(C['F2'], 't_cs') if C['F2'] else 'NA'}")
    res["C_fixed_horizon"] = C
    OUT[reg] = res

us, eu = OUT["USA"], OUT["EU24"]
def ov(a, b):
    return not (a["ci95_pp"][1] < b["ci95_pp"][0] or b["ci95_pp"][1] < a["ci95_pp"][0])
het = {}
for name in ("pluscat", "plusstage"):
    a, b = us["A_ladder"][name], eu["A_ladder"][name]
    het[name] = {"sign_differs": bool(np.sign(a["coef_pp"]) != np.sign(b["coef_pp"])), "ci_overlap": ov(a, b)}
cf_us, cf_eu = us["C_fixed_horizon"]["F2t"], eu["C_fixed_horizon"]["F2t"]
het["F2t"] = {"sign_differs": bool(cf_us and cf_eu and np.sign(cf_us["terrain"]["coef"]) != np.sign(cf_eu["terrain"]["coef"])),
              "ci_overlap": bool(cf_us and cf_eu and not (cf_us["terrain"]["ci95"][1] < cf_eu["terrain"]["ci95"][0] or cf_eu["terrain"]["ci95"][1] < cf_us["terrain"]["ci95"][0]))}
OUT["heterogeneity_flags"] = het
regional_het = any(v["sign_differs"] and not v["ci_overlap"] for v in het.values())
pred = {"US_pluscat_in_[-7,-2]": -7 <= us["A_ladder"]["pluscat"]["coef_pp"] <= -2, "US_mixed_120_145": 120 <= us["A_ladder"]["pluscat"]["n_mixed_cells"] <= 145,
        "EU_mixed_25_50": 25 <= eu["A_ladder"]["pluscat"]["n_mixed_cells"] <= 50, "EU_halfwidth_ge_8": (eu["A_ladder"]["pluscat"]["ci95_pp"][1] - eu["A_ladder"]["pluscat"]["ci95_pp"][0]) / 2 >= 8,
        "US_placebo_5_7": 5 <= us["A_ladder"]["pluscat"].get("placebo_p95_abs_pp", 0) <= 7, "EU_placebo_ge_10": eu["A_ladder"]["pluscat"].get("placebo_p95_abs_pp", 0) >= 10,
        "US_F2t_in_[0.20,0.35]_lower_gt0": bool(cf_us and 0.20 <= cf_us["terrain"]["coef"] <= 0.35 and cf_us["terrain"]["ci95"][0] > 0),
        "EU_F2t_incl0": bool(cf_eu and cf_eu["terrain"]["ci95"][0] <= 0 <= cf_eu["terrain"]["ci95"][1]),
        "B_fon_incl0_both": (not us["B_adjudication_stage_cell"]["fon"]["sig"]) and (not eu["B_adjudication_stage_cell"]["fon"]["sig"]),
        "NA_within_0.5pp_of_US": abs(OUT["NA"]["A_ladder"]["pluscat"]["coef_pp"] - us["A_ladder"]["pluscat"]["coef_pp"]) <= 0.5}
pred = {k: bool(v) for k, v in pred.items()}
OUT["prediction_check"] = pred
na = OUT["NA"]
verdict = (f"NA(미국+캐나다): cat 격차 {na['A_ladder']['pluscat']['coef_pp']:+.2f}pp [{na['A_ladder']['pluscat']['ci95_pp'][0]:+.1f},{na['A_ladder']['pluscat']['ci95_pp'][1]:+.1f}] · F2t {fmt(na['C_fixed_horizon']['F2t'], 'terrain') if na['C_fixed_horizon']['F2t'] else 'NA'} | "
           f"미국: cat 격차 {us['A_ladder']['pluscat']['coef_pp']:+.2f}pp [{us['A_ladder']['pluscat']['ci95_pp'][0]:+.1f},{us['A_ladder']['pluscat']['ci95_pp'][1]:+.1f}] (혼합 {us['A_ladder']['pluscat']['n_mixed_cells']} 셀/{us['A_ladder']['pluscat']['n_deals_mixed']} 딜, 위약 p95 {us['A_ladder']['pluscat'].get('placebo_p95_abs_pp')}) · +stage {us['A_ladder']['plusstage']['coef_pp']:+.2f} · fon {us['B_adjudication_stage_cell']['fon']['coef_pp']:+.2f} · F2t {fmt(cf_us, 'terrain') if cf_us else 'NA'} | "
           f"유럽: cat 격차 {eu['A_ladder']['pluscat']['coef_pp']:+.2f}pp [{eu['A_ladder']['pluscat']['ci95_pp'][0]:+.1f},{eu['A_ladder']['pluscat']['ci95_pp'][1]:+.1f}] (혼합 {eu['A_ladder']['pluscat']['n_mixed_cells']} 셀/{eu['A_ladder']['pluscat']['n_deals_mixed']} 딜, 위약 p95 {eu['A_ladder']['pluscat'].get('placebo_p95_abs_pp')}) · +stage {eu['A_ladder']['plusstage']['coef_pp']:+.2f} · fon {eu['B_adjudication_stage_cell']['fon']['coef_pp']:+.2f} · F2t {fmt(cf_eu, 'terrain') if cf_eu else 'NA'} — "
           f"{'지역 이질성(부호 상이·CI 비중첩)' if regional_het else '지역 이질성 미검출(부호 같음 또는 CI 중첩)'} (예측 적중 {sum(pred.values())}/{len(pred)})")
emit("P001-48", "지역 분할 로버스트니스: 미국 · NA(미국+캐나다) · 유럽 — 격차 사다리·판별 마진·고정지평 구성 계수", "OK", OUT,
     prediction="US cat ∈[−7,−2], 혼합 120–145, 위약 5–7; EU 혼합 25–50, 반폭≥8, 위약≥10; US F2t ∈[0.20,0.35] 하한>0; EU F2t 0 포함; fon 둘 다 0 포함",
     verdict=verdict, kill_met=False, n=int(us["n_deals"] + eu["n_deals"]),
     extra={"stage": 7, "feeds": "§7 로버스트니스 · Table 9 지역 분할 패널", "slug": "region_split", "builds_on": "P001-10/33/40", "common_sha256_16": COMMON_SHA})
log("done")
